# Advanced models — team summary

**Project:** NYC Yellow Taxi — trip duration & congestion fee prediction  
**Purpose:** One place to compare baselines vs ensemble models. Full training, tuning, and plots stay in each member’s notebook (linked below).

| Member | Notebook | Role |
|--------|----------|------|
| Abhishek Mohan Hundalekar | `Abhishek_RF.ipynb` | Random Forest starter (regression + classification) |
| Moses Kamya | `Moses_RF.ipynb` | Random Forest hyperparameter tuning (same pipeline as starter) |
| Morgan Cook | `Morgan_XGB.ipynb` | XGBoost classifier + regressor |
| Tarun Singh | `Tarun_XGB.ipynb` | XGBoost with train/val/test workflow, diagnostics |



## Shared setup 

- **Targets:** `trip_duration_min` (regression), `has_congestion_fee` (classification).
- **Leakage:** Drop post-trip payments, raw fare totals used as targets, and other leaky columns documented in `Abhishek_RF.ipynb` / `Moses_RF.ipynb`.
- **Encoding:** Categoricals label-encoded for tree models.
- **Common starter recipe:** 500,000-row sample with `random_state=42`, 80/20 train–test, stratify on congestion (Abhishek & Moses RF).
- **Milestone 6 next:** Residuals, ROC/AUC deep dive, and failure cases — see `work/06_model_evaluation/`.

## Test-set results — trip duration (regression)

Metrics are **minutes** (MAE, RMSE) and **R²** on the **test** portion. Values below are **representative** outputs from team notebooks; **re-run** your canonical pipeline and paste updated numbers before submission.

| Model | RMSE (min) | MAE (min) | R² | Source |
|-------|------------|-----------|-----|--------|
| Linear regression (baseline) | 5.45 | 3.75 | 0.739 | `Abhishek_Baseline.ipynb` |
| Random Forest (starter) | 3.99 | 2.61 | 0.858 | `Abhishek_RF.ipynb` |
| Random Forest (tuned, level target) | ~3.99 | ~2.61 | ~0.858 | `Moses_RF.ipynb`  |
| Random Forest (log1p target, expm1 pred.) | ~3.99 | ~2.57 | ~0.858 | `Moses_RF.ipynb`  |
| XGBoost regressor | ~4.13 | ~2.70 | ~0.849 | `Morgan_XGB.ipynb` |


**Takeaway:** Tree ensembles beat the linear baseline by a large margin on RMSE. RF and XGB are in the same ballpark; small differences depend on tuning and exact split/sample.

## Test-set results — congestion fee (classification)

| Model | Accuracy | ROC-AUC | F1 | Source |
|-------|----------|---------|-----|--------|
| Logistic regression (baseline, typical) | ~0.67 | ~0.73 | — | `Morgan_Baseline.ipynb` (100k sample) |
| Random Forest (starter) | ~0.953 | ~0.979 | ~0.968 | `Abhishek_RF.ipynb` |
| Random Forest (tuned) | ~0.964 | ~0.986 | ~0.975 | `Moses_RF.ipynb` |
| XGBoost classifier (Morgan) | ~0.962 | ~0.985 | — | `Morgan_XGB.ipynb` |
| XGBoost classifier (Tarun) | ~0.963 | ~0.986 | — | `Tarun_XGB.ipynb` |

**Takeaway:** Strong separation of fee vs no-fee; location-driven patterns. Tuned RF / XGB slightly improve over the RF starter on reported runs.

## Short conclusions 

- **Duration:** Use ensemble models for production-quality error vs linear regression; consider **log target** if the goal is to minimize **MAE** on skewed trip times.
- **Congestion:** Ensembles substantially outperform logistic baseline.



In [1]:

import pandas as pd

duration = pd.DataFrame([
    {"model": "Linear regression (baseline)", "RMSE_min": 5.45, "MAE_min": 3.75, "R2": 0.739, "notebook": "work/04_baseline_models/Abhishek_Baseline.ipynb"},
    {"model": "RF starter", "RMSE_min": 3.99, "MAE_min": 2.61, "R2": 0.858, "notebook": "Abhishek_RF.ipynb"},
    {"model": "RF tuned (level)", "RMSE_min": 3.99, "MAE_min": 2.61, "R2": 0.858, "notebook": "Moses_RF.ipynb"},
    {"model": "RF log1p→minutes", "RMSE_min": 3.99, "MAE_min": 2.57, "R2": 0.858, "notebook": "Moses_RF.ipynb"},
    {"model": "XGBoost (Morgan)", "RMSE_min": 4.13, "MAE_min": 2.70, "R2": 0.849, "notebook": "Morgan_XGB.ipynb"},
])

congestion = pd.DataFrame([
    {"model": "Logistic (baseline)", "accuracy": 0.669, "ROC_AUC": 0.734, "notebook": "Morgan_Baseline.ipynb"},
    {"model": "RF starter", "accuracy": 0.953, "ROC_AUC": 0.979, "notebook": "Abhishek_RF.ipynb"},
    {"model": "RF tuned", "accuracy": 0.964, "ROC_AUC": 0.986, "notebook": "Moses_RF.ipynb"},
    {"model": "XGBoost (Morgan)", "accuracy": 0.962, "ROC_AUC": 0.985, "notebook": "Morgan_XGB.ipynb"},
    {"model": "XGBoost (Tarun)", "accuracy": 0.963, "ROC_AUC": 0.986, "notebook": "Tarun_XGB.ipynb"},
])

print("Trip duration (test)")
print(duration.to_string(index=False))
print("\nCongestion fee (test)")
print(congestion.to_string(index=False))

Trip duration (test)
                       model  RMSE_min  MAE_min    R2                                        notebook
Linear regression (baseline)      5.45     3.75 0.739 work/04_baseline_models/Abhishek_Baseline.ipynb
                  RF starter      3.99     2.61 0.858                               Abhishek_RF.ipynb
            RF tuned (level)      3.99     2.61 0.858                                  Moses_RF.ipynb
            RF log1p→minutes      3.99     2.57 0.858                                  Moses_RF.ipynb
            XGBoost (Morgan)      4.13     2.70 0.849                                Morgan_XGB.ipynb

Congestion fee (test)
              model  accuracy  ROC_AUC              notebook
Logistic (baseline)     0.669    0.734 Morgan_Baseline.ipynb
         RF starter     0.953    0.979     Abhishek_RF.ipynb
           RF tuned     0.964    0.986        Moses_RF.ipynb
   XGBoost (Morgan)     0.962    0.985      Morgan_XGB.ipynb
    XGBoost (Tarun)     0.963    0.986 